# Retrieval-to-Extraction: From Documents to Structured JSON
### Building a Pipeline That Finds Relevant Docs and Extracts Typed Fields

**Project -- NLP / Information Extraction Portfolio**

---

Most LLM pipelines stop at *generation*: ask a question, get prose.
But production systems need **structured output** -- JSON records with
typed fields that downstream code can consume without parsing guesswork.

This notebook builds a full **retrieval-to-extraction** pipeline:

1. **Retrieve** the most relevant documents for a query
2. **Extract** structured fields (entities, categories, metrics) into JSON
3. **Validate** the extracted records against a schema

The architecture mirrors Haystack's `Pipeline` and LangChain's
`create_extraction_chain` patterns, explained from first principles.

## What You Will Learn

| Part | Topic | Why It Matters |
|---|---|---|
| 1 | The retrieval-to-extraction flow | Why retrieve *then* extract? |
| 2 | Build a document corpus | 30 technical incident reports |
| 3 | Build the retriever | BM25 + dense hybrid |
| 4 | Define extraction schemas | Pydantic-style JSON schemas |
| 5 | Build the extractor | Simulated structured-output LLM |
| 6 | Chain retrieval and extraction | End-to-end pipeline |
| 7 | Evaluate extraction quality | Precision, recall, schema compliance |
| 8 | Compare pipeline variants | 4 approaches side by side |
| 9 | Map to Haystack / LangChain | Production code |

## The Retrieval-to-Extraction Flow

```
     USER QUERY
     'Find incidents involving database outages in production'
         │
         ▼
  ┌──────────────┐
  │   RETRIEVER   │  Find the top-k most relevant incident reports
  │  (BM25+Dense) │  from the full corpus
  └──────────────┘
         │
         ▼   [doc_1, doc_2, doc_3]
  ┌──────────────┐
  │  EXTRACTOR    │  For each retrieved doc, extract typed fields
  │  (LLM + JSON) │  into a validated JSON record
  └──────────────┘
         │
         ▼   [{incident_type: 'database_outage', severity: 'P1', ...}, ...]
  ┌──────────────┐
  │  VALIDATOR    │  Check each record against the schema:
  │  (Schema)     │  required fields, allowed values, types
  └──────────────┘
         │
         ▼   Structured JSON output
```

### Why Not Extract From the Whole Corpus?

Extracting from every document is expensive and produces mostly irrelevant records.
Retrieval **focuses** the extraction step on a small, relevant subset of documents,
giving better accuracy and lower cost.

### Why Not Just Retrieve?

Retrieval returns raw text. Downstream systems need structured records.
Without extraction, you still need manual parsing or regex to consume the output.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, copy, hashlib, json as json_mod
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 2: Incident Report Corpus

30 synthetic technical incident reports. Each report describes a production
incident with embedded structured information: severity, service, root cause,
duration, impact, and remediation.

In [ ]:
@dataclass
class IncidentDoc:
    doc_id: str
    text: str
    metadata: dict
    embedding: Optional[np.ndarray] = None

INCIDENTS = [
    IncidentDoc('INC-001',
        'Production database outage on 2025-03-12 at 02:17 UTC. The primary PostgreSQL cluster '
        'in us-east-1 became unresponsive after a connection pool exhaustion caused by a runaway '
        'batch job. All API endpoints returned 503 errors for 47 minutes. The on-call DBA killed '
        'the offending connections and restarted pgbouncer. Root cause: missing connection limit '
        'on the analytics ETL service account. Impact: 12,400 failed API requests and $18,200 '
        'estimated revenue loss. Severity: P1. Remediation: connection pool limits added, '
        'alerting threshold lowered to 80% pool usage.',
        {'severity': 'P1', 'service': 'database', 'root_cause': 'connection_pool_exhaustion',
         'duration_min': 47, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-002',
        'Elevated error rate on the payment processing service starting 2025-04-05 14:30 UTC. '
        'Stripe webhook handler threw TimeoutError after Stripe changed their callback IP range '
        'and our firewall rules blocked the new addresses. Payment confirmations were delayed '
        'for 2 hours 15 minutes affecting 3,200 orders. Severity: P2. The networking team '
        'updated the IP allowlist. Root cause: stale firewall rules not updated from vendor '
        'changelog. Remediation: automated IP allowlist sync from Stripe API, weekly firewall audit.',
        {'severity': 'P2', 'service': 'payments', 'root_cause': 'firewall_misconfiguration',
         'duration_min': 135, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-003',
        'Complete loss of search functionality on 2025-02-18 09:00 UTC. Elasticsearch cluster '
        'entered red status after disk usage exceeded 95% on two data nodes. Search queries '
        'returned empty results for 1 hour 20 minutes. Severity: P1. Operations team expanded '
        'EBS volumes and rebalanced shards. Root cause: index retention policy was set to 90 '
        'days but should have been 30 days for log indices. Impact: search was unavailable '
        'during peak morning traffic, estimated 8,500 searches failed. Remediation: ILM policy '
        'corrected, disk usage alerting added at 80%.',
        {'severity': 'P1', 'service': 'search', 'root_cause': 'disk_exhaustion',
         'duration_min': 80, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-004',
        'CDN cache invalidation failure on 2025-05-22. Static assets served stale versions '
        'for 4 hours after a deployment. Users saw broken CSS and outdated JavaScript bundles. '
        'Severity: P3. The deploy pipeline did not trigger a CloudFront invalidation because '
        'the cache-busting hash was not updated in the CI config. Root cause: missing cache '
        'invalidation step in the deploy pipeline. Remediation: added automatic invalidation '
        'to the post-deploy hook and content-hash verification.',
        {'severity': 'P3', 'service': 'cdn', 'root_cause': 'cache_invalidation_failure',
         'duration_min': 240, 'region': 'global', 'incident_type': 'degradation'}),

    IncidentDoc('INC-005',
        'Authentication service returned 401 for all users on 2025-06-01 16:45 UTC. '
        'The JWT signing key rotation job ran early and invalidated all active tokens. '
        '100% of authenticated requests failed for 22 minutes. Severity: P1. '
        'Root cause: cron schedule misconfiguration rotated keys daily instead of monthly. '
        'Remediation: key rotation schedule corrected, grace period added for old keys, '
        'automated smoke test after rotation.',
        {'severity': 'P1', 'service': 'auth', 'root_cause': 'key_rotation_misconfiguration',
         'duration_min': 22, 'region': 'global', 'incident_type': 'outage'}),

    IncidentDoc('INC-006',
        'Memory leak in the recommendation engine detected on 2025-03-28. RSS grew from '
        '2 GB to 14 GB over 6 hours before OOM killer terminated the process. Recommendations '
        'were unavailable for 8 minutes during restart. Severity: P3. Root cause: unbounded '
        'in-memory cache in the collaborative filtering module. Remediation: LRU eviction '
        'policy added, memory alerts at 80% of container limit.',
        {'severity': 'P3', 'service': 'recommendation', 'root_cause': 'memory_leak',
         'duration_min': 8, 'region': 'us-west-2', 'incident_type': 'degradation'}),

    IncidentDoc('INC-007',
        'Notification service sent duplicate emails to 45,000 users on 2025-07-14. '
        'A Kafka consumer group rebalance caused message replay without idempotency checks. '
        'Users received 2-5 copies of the same promotional email. Severity: P2. '
        'Root cause: missing idempotency key in the email sending path. '
        'Remediation: idempotency keys added using message offset, deduplication window '
        'set to 24 hours.',
        {'severity': 'P2', 'service': 'notifications', 'root_cause': 'missing_idempotency',
         'duration_min': 0, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-008',
        'Load balancer health check misconfiguration caused 30% of traffic to be routed '
        'to unhealthy backend instances on 2025-04-19. Users experienced intermittent 502 '
        'errors for 1 hour 45 minutes. Severity: P2. Root cause: health check endpoint '
        'returned 200 even when the application was in a degraded state. Remediation: '
        'deep health check added that verifies database and cache connectivity.',
        {'severity': 'P2', 'service': 'load_balancer', 'root_cause': 'health_check_misconfiguration',
         'duration_min': 105, 'region': 'eu-west-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-009',
        'Data pipeline produced incorrect aggregations on 2025-08-03. A timezone '
        'conversion bug in the ETL job shifted daily revenue totals by 5 hours, '
        'causing financial reports to show wrong numbers for 3 days before detection. '
        'Severity: P2. Root cause: server timezone was changed from UTC to local time '
        'during a maintenance window. Remediation: all ETL jobs now use explicit UTC '
        'timestamps, timezone validation added to pipeline smoke tests.',
        {'severity': 'P2', 'service': 'data_pipeline', 'root_cause': 'timezone_bug',
         'duration_min': 4320, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-010',
        'Kubernetes cluster autoscaler failed to provision new nodes during a traffic '
        'spike on 2025-09-15. Pod scheduling latency exceeded 10 minutes. API response '
        'times degraded to 8 seconds average for 35 minutes. Severity: P2. '
        'Root cause: instance type requested was out of capacity in the availability zone. '
        'Remediation: added fallback instance types to the node group, pre-warmed capacity '
        'buffer of 2 nodes.',
        {'severity': 'P2', 'service': 'infrastructure', 'root_cause': 'capacity_exhaustion',
         'duration_min': 35, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-011',
        'Redis cluster failover caused 5-minute cache miss storm on 2025-05-10. '
        'The sentinel-triggered failover flushed all cached sessions, causing a thundering '
        'herd on the database. Database CPU peaked at 98% and response times spiked to '
        '12 seconds. Severity: P1. Root cause: sentinel failover did not preserve the '
        'replica dataset. Remediation: switched to Redis Cluster with hash slots for '
        'zero-downtime failover.',
        {'severity': 'P1', 'service': 'cache', 'root_cause': 'failover_data_loss',
         'duration_min': 5, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-012',
        'SSL certificate expired on the API gateway on 2025-06-30 00:00 UTC. '
        'All HTTPS connections were rejected with certificate errors for 1 hour 10 minutes '
        'until the on-call engineer manually renewed the certificate. Severity: P1. '
        'Root cause: certificate auto-renewal was disabled after a DNS provider migration. '
        'Remediation: cert-manager reinstalled, expiry monitoring with 30-day advance alerts.',
        {'severity': 'P1', 'service': 'api_gateway', 'root_cause': 'certificate_expiry',
         'duration_min': 70, 'region': 'global', 'incident_type': 'outage'}),

    IncidentDoc('INC-013',
        'Feature flag rollout caused a crash loop in the checkout service on 2025-07-22. '
        'A feature flag was enabled globally instead of for the 5% canary group. The new '
        'code path had an unhandled null reference. Pods restarted every 30 seconds for '
        '15 minutes. Severity: P1. Root cause: feature flag targeting rule was deleted '
        'instead of the flag being toggled. Remediation: added confirmation prompt for '
        'global flag changes, kill switch SLA of 2 minutes.',
        {'severity': 'P1', 'service': 'checkout', 'root_cause': 'feature_flag_misconfiguration',
         'duration_min': 15, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-014',
        'S3 bucket policy change exposed 2,300 internal documents publicly for 6 hours '
        'on 2025-04-28. A terraform apply removed the deny-public-access guardrail. '
        'No evidence of external access was found in CloudTrail logs. Severity: P1. '
        'Root cause: terraform state drift caused a destructive plan. Remediation: '
        'OPA policy gate added to CI to block public bucket policies, drift detection '
        'scheduled hourly.',
        {'severity': 'P1', 'service': 'storage', 'root_cause': 'iac_state_drift',
         'duration_min': 360, 'region': 'us-east-1', 'incident_type': 'security'}),

    IncidentDoc('INC-015',
        'GraphQL API returned PII in error messages on 2025-08-19. Stack traces included '
        'user email addresses and partial credit card numbers in verbose error responses '
        'for 12 hours. Severity: P1. Root cause: debug mode was left enabled after a '
        'staging environment merge. Impact: 340 unique users had PII exposed in API responses. '
        'Remediation: debug mode gated behind environment variable, PII scrubber added to '
        'error middleware.',
        {'severity': 'P1', 'service': 'api', 'root_cause': 'debug_mode_leak',
         'duration_min': 720, 'region': 'global', 'incident_type': 'security'}),

    IncidentDoc('INC-016',
        'Inventory sync job skipped 8,000 SKUs on 2025-09-02. A pagination bug in the '
        'warehouse API client caused the sync to stop at page 50 of 83. Stock levels '
        'were stale for 2 days before a merchant reported overselling. Severity: P2. '
        'Root cause: API client did not handle the new cursor-based pagination format. '
        'Remediation: pagination logic rewritten, end-of-data assertion added, daily '
        'SKU count reconciliation.',
        {'severity': 'P2', 'service': 'inventory', 'root_cause': 'pagination_bug',
         'duration_min': 2880, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-017',
        'Monitoring blind spot: Prometheus was not scraping the new order-processing '
        'microservice for 2 weeks after its launch on 2025-06-10. A latency regression '
        'went undetected until customer complaints escalated. Severity: P3. '
        'Root cause: service discovery label selector was not updated for the new namespace. '
        'Remediation: onboarding checklist now includes monitoring verification, namespace '
        'coverage dashboard added.',
        {'severity': 'P3', 'service': 'monitoring', 'root_cause': 'monitoring_gap',
         'duration_min': 20160, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-018',
        'DNS propagation delay caused intermittent failures on 2025-05-05 after a '
        'migration from Route 53 to Cloudflare. TTL was set to 3600 seconds but some '
        'recursive resolvers cached the old record for up to 48 hours. 15% of users '
        'experienced connection failures. Severity: P2. Root cause: TTL was not lowered '
        'before the migration. Remediation: pre-migration TTL reduction to 60s, staged '
        'migration with gradual traffic shifting.',
        {'severity': 'P2', 'service': 'dns', 'root_cause': 'dns_propagation_delay',
         'duration_min': 2880, 'region': 'global', 'incident_type': 'degradation'}),

    IncidentDoc('INC-019',
        'Clickstream data loss on 2025-07-30. The Kafka retention period was shortened '
        'from 7 days to 1 day during a cost reduction exercise. A consumer lag caused '
        'the analytics pipeline to miss 36 hours of click events. Severity: P2. '
        'Root cause: retention change was applied without checking consumer lag. '
        'Remediation: consumer lag alerting added, retention changes require sign-off '
        'from data engineering.',
        {'severity': 'P2', 'service': 'streaming', 'root_cause': 'retention_misconfiguration',
         'duration_min': 2160, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-020',
        'CI/CD pipeline deployed an untested branch to production on 2025-08-12. '
        'A merge conflict resolution left the deployment branch pointing to a feature '
        'branch. The release contained experimental pricing logic that discounted all '
        'items by 90%. Detected after 25 minutes when anomaly detection flagged '
        'abnormal order values. Severity: P1. Root cause: branch protection rules '
        'were bypassed using admin override. Remediation: removed admin bypass, '
        'added pre-deploy diff review gate.',
        {'severity': 'P1', 'service': 'cicd', 'root_cause': 'branch_protection_bypass',
         'duration_min': 25, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-021',
        'API rate limiter blocked legitimate traffic on 2025-09-08. A misconfigured '
        'rate limit rule set the threshold to 10 requests per minute instead of 1000. '
        'Enterprise API customers were throttled for 3 hours. Severity: P2. '
        'Root cause: YAML config used wrong unit (reqs/min instead of reqs/sec). '
        'Remediation: config validation added, rate limit changes require canary period.',
        {'severity': 'P2', 'service': 'api_gateway', 'root_cause': 'rate_limit_misconfiguration',
         'duration_min': 180, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-022',
        'Database migration caused 45-minute downtime on 2025-03-20. An ALTER TABLE '
        'statement locked the orders table while adding a NOT NULL column with a '
        'default value. All writes to the orders service failed. Severity: P1. '
        'Root cause: migration was not tested against production-scale data. '
        'Remediation: all DDL changes require pt-online-schema-change, migration '
        'dry-run on production-sized replica.',
        {'severity': 'P1', 'service': 'database', 'root_cause': 'blocking_migration',
         'duration_min': 45, 'region': 'us-east-1', 'incident_type': 'outage'}),

    IncidentDoc('INC-023',
        'Log ingestion pipeline dropped 40% of logs on 2025-06-18. The Fluentd buffer '
        'overflowed after a noisy deployment generated 10x normal log volume. '
        'Investigation into a separate billing anomaly was delayed because the relevant '
        'logs were missing. Severity: P3. Root cause: fixed-size buffer without backpressure. '
        'Remediation: switched to persistent disk buffer, added log volume anomaly alerting.',
        {'severity': 'P3', 'service': 'logging', 'root_cause': 'buffer_overflow',
         'duration_min': 480, 'region': 'us-west-2', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-024',
        'Third-party geocoding API outage on 2025-04-11 broke address validation in '
        'the checkout flow. Orders with international addresses could not be placed '
        'for 2 hours. Severity: P2. Root cause: no fallback geocoding provider was '
        'configured. 1,200 orders were abandoned. Remediation: added fallback to '
        'Nominatim with graceful degradation, circuit breaker on the primary provider.',
        {'severity': 'P2', 'service': 'checkout', 'root_cause': 'third_party_outage',
         'duration_min': 120, 'region': 'global', 'incident_type': 'degradation'}),

    IncidentDoc('INC-025',
        'Cron job race condition on 2025-07-05. Two instances of the billing '
        'reconciliation job ran simultaneously, double-charging 850 customers. '
        'Detected after 4 hours when customer support tickets spiked. Severity: P1. '
        'Root cause: cron job did not acquire a distributed lock. '
        'Remediation: distributed lock via Redis SETNX, job idempotency with '
        'deduplication key on invoice ID.',
        {'severity': 'P1', 'service': 'billing', 'root_cause': 'race_condition',
         'duration_min': 240, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-026',
        'Latency spike on the product catalog API on 2025-08-25. A new database index '
        'was being built online and consumed 70% of IOPS. P95 latency rose from 120ms '
        'to 4.2 seconds for 90 minutes. Severity: P3. Root cause: CREATE INDEX '
        'CONCURRENTLY was not used, causing I/O contention. Remediation: index builds '
        'scheduled off-peak, IOPS budget alerting added.',
        {'severity': 'P3', 'service': 'database', 'root_cause': 'index_build_contention',
         'duration_min': 90, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-027',
        'Webhook delivery failure on 2025-05-30. The webhook service ran out of '
        'outbound connections because the HTTP client was not reusing connections. '
        '28,000 webhooks were queued and delivered with up to 6-hour delays. Severity: P2. '
        'Root cause: HTTP client created a new TCP connection per request instead of '
        'using a connection pool. Remediation: connection pooling enabled, backlog '
        'monitoring added.',
        {'severity': 'P2', 'service': 'webhooks', 'root_cause': 'connection_exhaustion',
         'duration_min': 360, 'region': 'us-east-1', 'incident_type': 'degradation'}),

    IncidentDoc('INC-028',
        'Image processing service corrupted thumbnails on 2025-09-18. A library update '
        'changed the default JPEG quality from 85 to 20, producing visibly degraded '
        'product images. 15,000 listings were affected before detection. Severity: P2. '
        'Root cause: dependency version was not pinned. Remediation: pinned all image '
        'processing dependencies, visual regression tests added to CI.',
        {'severity': 'P2', 'service': 'media', 'root_cause': 'dependency_regression',
         'duration_min': 1440, 'region': 'us-east-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-029',
        'Cross-region replication lag on 2025-06-25. DynamoDB global table replication '
        'fell behind by 45 minutes during a bulk import. Users in eu-west-1 saw stale '
        'inventory data, leading to overselling of 120 items. Severity: P2. '
        'Root cause: bulk import exceeded the provisioned WCU, causing throttling. '
        'Remediation: bulk imports use on-demand capacity mode, replication lag alerting '
        'with 5-minute threshold.',
        {'severity': 'P2', 'service': 'database', 'root_cause': 'replication_lag',
         'duration_min': 45, 'region': 'eu-west-1', 'incident_type': 'data_quality'}),

    IncidentDoc('INC-030',
        'Scheduled maintenance window overran on 2025-10-01. A planned 30-minute '
        'database upgrade took 2 hours 40 minutes due to unexpected data migration '
        'requirements. The maintenance page was shown to all users during this period. '
        'Severity: P2. Root cause: migration script was not tested on production-scale '
        'data. Remediation: all maintenance scripts must pass a dry-run on a '
        'production-sized staging environment.',
        {'severity': 'P2', 'service': 'database', 'root_cause': 'maintenance_overrun',
         'duration_min': 160, 'region': 'us-east-1', 'incident_type': 'outage'}),
]

print(f'Corpus: {len(INCIDENTS)} incident reports')
sev = Counter(d.metadata['severity'] for d in INCIDENTS)
print(f'Severities: {dict(sev)}')
types = Counter(d.metadata['incident_type'] for d in INCIDENTS)
print(f'Types: {dict(types)}')

## Part 3: Building the Retriever

Hybrid BM25 + dense retrieval, the same proven pattern used in
Haystack's `DocumentStore` + `HybridRetriever`.

In [ ]:
class TinyEmbedder:
    def __init__(self, dim=96):
        self.dim = dim
        self.vocab = {}
        self.idf = {}
        self.proj = None

    def fit(self, texts):
        df = Counter()
        for t in texts:
            for tok in set(re.findall(r'[a-z0-9]+', t.lower())):
                df[tok] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self.proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def encode(self, text):
        vec = np.zeros(len(self.vocab))
        for tok, cnt in Counter(re.findall(r'[a-z0-9]+', text.lower())).items():
            if tok in self.vocab:
                vec[self.vocab[tok]] = cnt * self.idf.get(tok, 1.0)
        proj = vec @ self.proj
        norm = np.linalg.norm(proj)
        return proj / norm if norm > 0 else proj


emb = TinyEmbedder(96)
emb.fit([d.text for d in INCIDENTS])
for d in INCIDENTS:
    d.embedding = emb.encode(d.text)


class HybridRetriever:
    def __init__(self, docs, embedder, k1=1.5, b=0.75, k_rrf=60):
        self.docs = docs
        self.embedder = embedder
        self.k1, self.b, self.k_rrf = k1, b, k_rrf
        self._toks = {d.doc_id: re.findall(r'[a-z0-9]+', d.text.lower()) for d in docs}
        self._df = Counter()
        for toks in self._toks.values():
            for t in set(toks):
                self._df[t] += 1
        self._avgdl = float(np.mean([len(t) for t in self._toks.values()]))
        self._n = len(docs)
        self._mat = np.stack([d.embedding for d in docs])
        self._id_map = {d.doc_id: d for d in docs}

    def _bm25(self, query, top_k):
        qtoks = re.findall(r'[a-z0-9]+', query.lower())
        scores = {}
        for d in self.docs:
            toks = self._toks[d.doc_id]
            tf = Counter(toks)
            dl = len(toks)
            s = 0.0
            for qt in qtoks:
                f = tf.get(qt, 0)
                df = self._df.get(qt, 0)
                idf_val = math.log((self._n - df + 0.5) / (df + 0.5) + 1)
                tfn = f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self._avgdl))
                s += idf_val * tfn
            if s > 0:
                scores[d.doc_id] = s
        return sorted(scores.items(), key=lambda x: -x[1])[:top_k]

    def _dense(self, query, top_k):
        qe = self.embedder.encode(query)
        sims = self._mat @ qe
        idxs = np.argsort(sims)[::-1][:top_k]
        return [(self.docs[i].doc_id, float(sims[i])) for i in idxs]

    def retrieve(self, query, top_k=5):
        bm25_r = self._bm25(query, top_k * 3)
        dense_r = self._dense(query, top_k * 3)
        rrf = defaultdict(float)
        for rank, (did, _) in enumerate(bm25_r):
            rrf[did] += 1.0 / (self.k_rrf + rank + 1)
        for rank, (did, _) in enumerate(dense_r):
            rrf[did] += 1.0 / (self.k_rrf + rank + 1)
        ranked = sorted(rrf.items(), key=lambda x: -x[1])[:top_k]
        return [self._id_map[did] for did, _ in ranked]


retriever = HybridRetriever(INCIDENTS, emb)
print('Hybrid retriever ready')

# Quick test
test_q = 'database outage in production'
test_hits = retriever.retrieve(test_q, top_k=3)
for d in test_hits:
    print(f'  {d.doc_id}: {d.text[:80]}...')

## Part 4: Extraction Schema

We define the JSON schema that the extractor must produce for each document.
This mirrors Pydantic model definitions used by LangChain's extraction chains
and Haystack's `OutputAdapter`.

```python
# Real Pydantic version (LangChain / Haystack)
from pydantic import BaseModel, Field
from typing import Literal

class IncidentRecord(BaseModel):
    incident_id: str
    severity: Literal['P1', 'P2', 'P3', 'P4']
    service: str
    incident_type: Literal['outage', 'degradation', 'security', 'data_quality']
    root_cause: str
    duration_minutes: int
    region: str
    remediation: str
```

In [ ]:
EXTRACTION_SCHEMA = {
    'incident_id': {'type': 'string', 'required': True},
    'severity': {'type': 'string', 'required': True,
                 'allowed': ['P1', 'P2', 'P3', 'P4']},
    'service': {'type': 'string', 'required': True},
    'incident_type': {'type': 'string', 'required': True,
                      'allowed': ['outage', 'degradation', 'security', 'data_quality']},
    'root_cause': {'type': 'string', 'required': True},
    'duration_minutes': {'type': 'integer', 'required': True},
    'region': {'type': 'string', 'required': True},
    'remediation': {'type': 'string', 'required': False},
}

print('Extraction schema:')
for field_name, spec in EXTRACTION_SCHEMA.items():
    allowed = f" -- allowed: {spec['allowed']}" if 'allowed' in spec else ''
    req = 'required' if spec.get('required') else 'optional'
    print(f'  {field_name:<20s}: {spec["type"]:<10s} ({req}){allowed}')

## Part 5: Building the Extractor

The extractor reads a document and fills the schema fields.
In production, this is an LLM with structured output (JSON mode / function calling).
Here, we simulate extraction with deterministic parsing.

### How extraction quality varies

We model four quality levels:

| Level | Description | Simulates |
|---|---|---|
| `regex_only` | Rigid pattern matching | No LLM, baseline |
| `basic_llm` | LLM with basic prompt | GPT-3.5 / small model |
| `guided_llm` | LLM with schema guidance + CoT | GPT-4 / tuned model |
| `guided_with_demos` | + few-shot demonstrations | Optimised extraction |

In [ ]:
STOPWORDS = {'the','a','an','is','are','was','were','be','been','being',
             'have','has','had','do','does','did','will','would','could',
             'should','may','might','can','shall','to','of','in','for',
             'on','with','at','by','from','as','into','about','between',
             'through','during','and','but','or','nor','not','so','yet',
             'if','then','it','its','that','this','than','after','all'}


class StructuredExtractor:
    SEVERITY_PAT = re.compile(r'severity[:\s]+(p[1-4])', re.I)
    DURATION_PAT = re.compile(r'(\d+)\s*(?:minutes?|min)', re.I)
    DURATION_HR_PAT = re.compile(r'(\d+)\s*hours?\s*(\d+)?\s*(?:minutes?|min)?', re.I)
    REGION_PAT = re.compile(r'(us-east-1|us-west-2|eu-west-1|ap-southeast-1|global)', re.I)

    TYPE_KEYWORDS = {
        'outage': ['outage', 'unavailable', 'down', 'unresponsive', '503', 'crash loop',
                   'failed for', 'rejected', 'downtime'],
        'degradation': ['degraded', 'degradation', 'elevated error', 'latency',
                        'intermittent', 'delayed', 'slow', 'throttled', 'stale'],
        'security': ['exposed', 'pii', 'public', 'security', 'breach', 'unauthorized'],
        'data_quality': ['incorrect', 'duplicate', 'data loss', 'corrupted', 'wrong',
                         'skipped', 'missing', 'dropped', 'double', 'oversell'],
    }

    def __init__(self, quality='guided_llm'):
        self.quality = quality
        self._seed = 42

    def _rng(self):
        self._seed = (self._seed * 6364136223846793005 + 1) & 0xFFFFFFFF
        return (self._seed >> 16) / 65535.0

    def _extract_severity(self, text):
        m = self.SEVERITY_PAT.search(text)
        return m.group(1).upper() if m else 'P3'

    def _extract_duration(self, text):
        # Try 'X hours Y minutes' first
        m = self.DURATION_HR_PAT.search(text)
        if m:
            hrs = int(m.group(1))
            mins = int(m.group(2)) if m.group(2) else 0
            return hrs * 60 + mins
        m = self.DURATION_PAT.search(text)
        if m:
            return int(m.group(1))
        if 'days' in text.lower() or 'day' in text.lower():
            dm = re.search(r'(\d+)\s*days?', text, re.I)
            if dm:
                return int(dm.group(1)) * 1440
        return 0

    def _extract_region(self, text):
        m = self.REGION_PAT.search(text)
        return m.group(1).lower() if m else 'unknown'

    def _extract_type(self, text):
        tl = text.lower()
        scores = {}
        for itype, kws in self.TYPE_KEYWORDS.items():
            scores[itype] = sum(1 for kw in kws if kw in tl)
        best = max(scores, key=scores.get)
        return best if scores[best] > 0 else 'degradation'

    def _extract_service(self, text):
        tl = text.lower()
        service_kws = {
            'database': ['database', 'postgresql', 'dynamodb', 'sql', 'migration'],
            'payments': ['payment', 'stripe', 'billing', 'charge'],
            'search': ['elasticsearch', 'search'],
            'cdn': ['cdn', 'cloudfront', 'cache invalidation'],
            'auth': ['authentication', 'jwt', 'token', 'signing key'],
            'checkout': ['checkout', 'order', 'feature flag'],
            'notifications': ['notification', 'email'],
            'api_gateway': ['api gateway', 'ssl certificate', 'rate limit'],
            'infrastructure': ['kubernetes', 'autoscaler', 'node'],
            'cache': ['redis', 'cache'],
            'storage': ['s3', 'bucket'],
            'api': ['graphql', 'api'],
            'inventory': ['inventory', 'sku'],
            'monitoring': ['prometheus', 'monitoring'],
            'dns': ['dns', 'route 53', 'cloudflare'],
            'streaming': ['kafka', 'clickstream'],
            'cicd': ['ci/cd', 'pipeline', 'deploy'],
            'logging': ['log', 'fluentd'],
            'webhooks': ['webhook'],
            'media': ['image', 'thumbnail'],
            'recommendation': ['recommendation'],
            'data_pipeline': ['etl', 'data pipeline', 'aggregation'],
            'billing': ['billing', 'reconciliation', 'invoice'],
            'load_balancer': ['load balancer', 'health check'],
        }
        best, best_score = 'unknown', 0
        for svc, kws in service_kws.items():
            score = sum(1 for kw in kws if kw in tl)
            if score > best_score:
                best, best_score = svc, score
        return best

    def _extract_root_cause(self, text):
        # Find the sentence containing 'root cause'
        sents = re.split(r'(?<=[.!?])\s+', text)
        for s in sents:
            if 'root cause' in s.lower():
                # Extract the explanation after the colon
                parts = s.split(':')
                if len(parts) >= 2:
                    cause = parts[-1].strip().rstrip('.')
                    return cause
                return s.strip()
        return 'unknown'

    def _extract_remediation(self, text):
        sents = re.split(r'(?<=[.!?])\s+', text)
        for s in sents:
            if 'remediation' in s.lower():
                parts = s.split(':')
                if len(parts) >= 2:
                    return parts[-1].strip().rstrip('.')
                return s.strip()
        return ''

    def extract(self, doc):
        text = doc.text
        record = {
            'incident_id': doc.doc_id,
            'severity': self._extract_severity(text),
            'service': self._extract_service(text),
            'incident_type': self._extract_type(text),
            'root_cause': self._extract_root_cause(text),
            'duration_minutes': self._extract_duration(text),
            'region': self._extract_region(text),
            'remediation': self._extract_remediation(text),
        }

        # Quality-dependent noise
        if self.quality == 'regex_only':
            # Pure regex: loses nuance on type and service
            if self._rng() < 0.25:
                record['incident_type'] = 'degradation'
            if self._rng() < 0.15:
                record['service'] = 'unknown'
            record['root_cause'] = record['root_cause'][:60]
        elif self.quality == 'basic_llm':
            if self._rng() < 0.10:
                record['incident_type'] = 'degradation'
            if self._rng() < 0.08:
                record['severity'] = 'P2'
        elif self.quality == 'guided_llm':
            if self._rng() < 0.05:
                record['severity'] = 'P2'
        # guided_with_demos: no noise

        return record


extractor = StructuredExtractor('guided_llm')
sample = extractor.extract(INCIDENTS[0])
print('Sample extraction (INC-001):')
print(json_mod.dumps(sample, indent=2))

### Schema Validation

Every extracted record is validated against the schema before being accepted.
This is the structured-output guarantee that makes the pipeline production-ready.

In [ ]:
def validate_record(record, schema):
    errors = []
    for field_name, spec in schema.items():
        if spec.get('required') and field_name not in record:
            errors.append(f'Missing required field: {field_name}')
            continue
        if field_name not in record:
            continue
        val = record[field_name]
        if spec['type'] == 'string' and not isinstance(val, str):
            errors.append(f'{field_name}: expected string, got {type(val).__name__}')
        if spec['type'] == 'integer' and not isinstance(val, int):
            errors.append(f'{field_name}: expected int, got {type(val).__name__}')
        if 'allowed' in spec and val not in spec['allowed']:
            errors.append(f'{field_name}: {val!r} not in {spec["allowed"]}')
    return errors


# Validate the sample
errs = validate_record(sample, EXTRACTION_SCHEMA)
print(f'Validation errors: {errs if errs else "None -- record is valid"}')

## Part 6: End-to-End Pipeline

Chain retrieval and extraction into one pipeline call.

```
query -> retrieve top-k docs -> extract JSON from each -> validate -> output
```

This mirrors:
- Haystack: `Pipeline().add_component(retriever).add_component(extractor)`
- LangChain: `retriever | extraction_chain | validator`

In [ ]:
class RetrievalExtractionPipeline:
    def __init__(self, retriever, extractor, schema, top_k=5, name='Pipeline'):
        self.retriever = retriever
        self.extractor = extractor
        self.schema = schema
        self.top_k = top_k
        self.name = name

    def run(self, query):
        docs = self.retriever.retrieve(query, top_k=self.top_k)
        records = []
        for d in docs:
            rec = self.extractor.extract(d)
            errors = validate_record(rec, self.schema)
            records.append({
                'doc_id': d.doc_id,
                'record': rec,
                'valid': len(errors) == 0,
                'errors': errors,
            })
        return {
            'query': query,
            'n_retrieved': len(docs),
            'n_valid': sum(1 for r in records if r['valid']),
            'records': records,
        }


pipe = RetrievalExtractionPipeline(retriever, extractor, EXTRACTION_SCHEMA,
                                   top_k=5, name='Guided LLM Pipeline')

result = pipe.run('database outage incidents in production')
print(f'Query: {result["query"]}')
print(f'Retrieved: {result["n_retrieved"]} | Valid records: {result["n_valid"]}')
print()
for r in result['records']:
    print(f'{r["doc_id"]}: valid={r["valid"]}')
    print(f'  {json_mod.dumps(r["record"], indent=2)[:200]}')
    print()

## Part 7: Evaluation

We measure three things:

| Metric | What It Measures | Target Component |
|---|---|---|
| Retrieval recall | Did we find the relevant docs? | Retriever |
| Field accuracy | Does the extracted field match ground truth? | Extractor |
| Schema compliance | Is the output valid JSON per schema? | Validator |

These are measured **independently** so you know what to fix.

In [ ]:
EVAL_QUERIES = [
    {'query': 'database outages', 'expected_ids': {'INC-001', 'INC-022', 'INC-030'}},
    {'query': 'security incidents with data exposure', 'expected_ids': {'INC-014', 'INC-015'}},
    {'query': 'payment processing failures', 'expected_ids': {'INC-002', 'INC-025'}},
    {'query': 'data quality issues with incorrect or missing data', 'expected_ids': {'INC-009', 'INC-016', 'INC-019'}},
    {'query': 'certificate or authentication failures', 'expected_ids': {'INC-005', 'INC-012'}},
    {'query': 'Kafka or streaming incidents', 'expected_ids': {'INC-007', 'INC-019'}},
    {'query': 'infrastructure scaling or capacity problems', 'expected_ids': {'INC-010', 'INC-026'}},
]


def evaluate_pipeline(pipeline, queries, all_docs):
    rows = []
    for q in queries:
        result = pipeline.run(q['query'])
        retrieved_ids = {r['doc_id'] for r in result['records']}
        expected = q['expected_ids']

        # Retrieval recall
        recall = len(retrieved_ids & expected) / len(expected) if expected else 0.0

        # Field accuracy: compare extracted fields to ground truth for retrieved docs
        field_correct = 0
        field_total = 0
        for r in result['records']:
            gt_doc = next((d for d in all_docs if d.doc_id == r['doc_id']), None)
            if gt_doc:
                gt = gt_doc.metadata
                rec = r['record']
                for f in ['severity', 'service', 'incident_type']:
                    field_total += 1
                    if rec.get(f) == gt.get(f):
                        field_correct += 1

        field_acc = field_correct / field_total if field_total > 0 else 0.0

        # Schema compliance
        n_valid = result['n_valid']
        n_total = len(result['records'])
        compliance = n_valid / n_total if n_total > 0 else 0.0

        composite = (recall + field_acc + compliance) / 3

        rows.append({
            'pipeline': pipeline.name,
            'query': q['query'][:50],
            'recall': round(recall, 4),
            'field_accuracy': round(field_acc, 4),
            'schema_compliance': round(compliance, 4),
            'composite': round(composite, 4),
        })
    return pd.DataFrame(rows)


print('Evaluation framework ready')

## Part 8: Pipeline Variant Comparison

We compare four extraction quality levels, all using the same retriever.

In [ ]:
variants = [
    ('Regex Only', 'regex_only'),
    ('Basic LLM', 'basic_llm'),
    ('Guided LLM', 'guided_llm'),
    ('Guided + Demos', 'guided_with_demos'),
]

all_results = []
for name, quality in variants:
    ext = StructuredExtractor(quality)
    p = RetrievalExtractionPipeline(retriever, ext, EXTRACTION_SCHEMA,
                                    top_k=5, name=name)
    df = evaluate_pipeline(p, EVAL_QUERIES, INCIDENTS)
    all_results.append(df)

eval_df = pd.concat(all_results, ignore_index=True)
summary = eval_df.groupby('pipeline')[['recall', 'field_accuracy',
                                        'schema_compliance', 'composite']].mean().round(4)
summary = summary.reindex(['Regex Only', 'Basic LLM', 'Guided LLM', 'Guided + Demos'])
print('PIPELINE COMPARISON')
print('=' * 70)
print(summary.to_string())

# Lift
base_composite = summary.loc['Regex Only', 'composite']
print(f'\nLift over Regex Only (composite):')
for name in ['Basic LLM', 'Guided LLM', 'Guided + Demos']:
    lift = summary.loc[name, 'composite'] - base_composite
    print(f'  {name:<18s}: +{lift:.4f}')

## Visualisations

In [ ]:
PIPE_COLORS = {
    'Regex Only': '#e74c3c',
    'Basic LLM': '#e67e22',
    'Guided LLM': '#3498db',
    'Guided + Demos': '#2ecc71',
}

# Bar comparison
plot_df = summary.reset_index().melt(id_vars='pipeline', var_name='metric', value_name='score')
fig = px.bar(plot_df, x='metric', y='score', color='pipeline', barmode='group',
             color_discrete_map=PIPE_COLORS,
             title='Extraction Pipeline Comparison',
             template='plotly_white', height=430)
fig.update_yaxes(range=[0, 1])
fig.show()

In [ ]:
# Radar
dims = ['recall', 'field_accuracy', 'schema_compliance']
fig = go.Figure()
for name in summary.index:
    vals = [summary.loc[name, d] for d in dims]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=dims + [dims[0]],
        name=name, fill='toself',
        line_color=PIPE_COLORS.get(name, '#95a5a6'), opacity=0.55,
    ))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 1], visible=True)),
                  title='Quality Radar', template='plotly_white', height=480)
fig.show()

In [ ]:
# Trajectory
traj = [(name, summary.loc[name, 'composite']) for name in summary.index]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(traj))), y=[v for _, v in traj],
    mode='lines', line=dict(color='#bdc3c7', width=2, dash='dot'), showlegend=False,
))
for i, (label, val) in enumerate(traj):
    fig.add_trace(go.Scatter(
        x=[i], y=[val], mode='markers+text',
        marker=dict(size=18, color=PIPE_COLORS.get(label, '#95a5a6'),
                    line=dict(color='white', width=2)),
        text=[f'{val:.1%}'], textposition='top center', showlegend=False,
    ))
fig.update_layout(
    title='Composite Score Trajectory',
    xaxis=dict(tickmode='array', tickvals=list(range(len(traj))),
               ticktext=[t[0] for t in traj]),
    yaxis_title='Composite', yaxis_range=[0, 1],
    template='plotly_white', height=400,
)
fig.show()

In [ ]:
# Corpus composition
sev_df = pd.DataFrame(Counter(d.metadata['severity'] for d in INCIDENTS).items(),
                      columns=['severity', 'count']).sort_values('severity')
type_df = pd.DataFrame(Counter(d.metadata['incident_type'] for d in INCIDENTS).items(),
                       columns=['type', 'count']).sort_values('count', ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Severity Distribution', 'Incident Types'],
                    specs=[[{'type': 'pie'}, {'type': 'bar'}]])
fig.add_trace(go.Pie(labels=sev_df['severity'], values=sev_df['count'],
                     marker_colors=['#e74c3c', '#e67e22', '#f1c40f']),
              row=1, col=1)
fig.add_trace(go.Bar(x=type_df['type'], y=type_df['count'],
                     marker_color='#3498db'), row=1, col=2)
fig.update_layout(template='plotly_white', height=380, showlegend=False)
fig.show()

In [ ]:
# Per-query recall heatmap
hm = eval_df.pivot_table(index='query', columns='pipeline', values='recall')
hm = hm[['Regex Only', 'Basic LLM', 'Guided LLM', 'Guided + Demos']]
fig = px.imshow(hm.values, x=list(hm.columns), y=list(hm.index),
                color_continuous_scale='YlOrRd', text_auto='.2f',
                title='Retrieval Recall by Query and Pipeline',
                labels=dict(x='Pipeline', y='Query', color='Recall'))
fig.update_layout(template='plotly_white', height=400)
fig.show()

In [ ]:
# Field accuracy by field
field_acc_rows = []
for name, quality in variants:
    ext = StructuredExtractor(quality)
    for d in INCIDENTS:
        rec = ext.extract(d)
        gt = d.metadata
        for f in ['severity', 'service', 'incident_type']:
            field_acc_rows.append({
                'pipeline': name, 'field': f,
                'correct': 1 if rec.get(f) == gt.get(f) else 0,
            })

field_df = pd.DataFrame(field_acc_rows)
field_agg = field_df.groupby(['pipeline', 'field'])['correct'].mean().reset_index()

fig = px.bar(field_agg, x='field', y='correct', color='pipeline', barmode='group',
             color_discrete_map=PIPE_COLORS,
             title='Field-Level Extraction Accuracy',
             labels={'correct': 'Accuracy', 'field': 'Field'},
             template='plotly_white', height=420)
fig.update_yaxes(range=[0, 1])
fig.show()

## Part 9: Mapping to Haystack and LangChain

### Haystack version

```python
from haystack import Pipeline
from haystack.components.retrievers import InMemoryBM25Retriever
from haystack.components.generators import OllamaGenerator
from haystack.components.builders import PromptBuilder
from haystack.document_stores.in_memory import InMemoryDocumentStore

store = InMemoryDocumentStore()
store.write_documents(documents)

template = '''
Extract from the following incident report:
{{ document.content }}

Return JSON with: severity, service, incident_type, root_cause,
duration_minutes, region, remediation
'''

pipe = Pipeline()
pipe.add_component('retriever', InMemoryBM25Retriever(store))
pipe.add_component('prompt', PromptBuilder(template=template))
pipe.add_component('llm', OllamaGenerator(model='qwen3'))
pipe.connect('retriever.documents', 'prompt.document')
pipe.connect('prompt.prompt', 'llm.prompt')

result = pipe.run({'retriever': {'query': 'database outages'}})
```

### LangChain version

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

class IncidentRecord(BaseModel):
    severity: Literal['P1', 'P2', 'P3', 'P4']
    service: str
    incident_type: Literal['outage', 'degradation', 'security', 'data_quality']
    root_cause: str
    duration_minutes: int
    region: str
    remediation: str = ''

parser = JsonOutputParser(pydantic_object=IncidentRecord)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Extract structured incident data from the report. '
               '{format_instructions}'),
    ('human', '{document}'),
])

llm = ChatOllama(model='qwen3')
chain = prompt | llm | parser

# With retrieval:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5')
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

# Full pipeline: retrieve then extract
retrieved = retriever.invoke('database outage')
records = [chain.invoke({
    'document': doc.page_content,
    'format_instructions': parser.get_format_instructions()
}) for doc in retrieved]
```

### Architecture Comparison

| Feature | Our Teaching Pipeline | Haystack | LangChain |
|---|---|---|---|
| Retriever | BM25 + Dense hybrid | `InMemoryBM25Retriever` / `EmbeddingRetriever` | `Chroma.as_retriever()` |
| Extractor | Rule-based + simulated LLM | `OllamaGenerator` + PromptBuilder | `ChatOllama` + `JsonOutputParser` |
| Schema | Dict schema + validator | Haystack component contracts | Pydantic models |
| Validation | Manual field checks | Output adapters | `JsonOutputParser` / Pydantic |
| Pipeline wiring | Python glue code | `Pipeline().connect()` | LCEL `|` chains |

## Summary

### The retrieval-to-extraction pattern

```
query  ->  retrieve relevant docs  ->  extract JSON fields  ->  validate  ->  records
```

### Key Takeaways

| # | Insight |
|---|---|
| 1 | **Retrieve first, then extract** -- extraction on the full corpus is wasteful |
| 2 | **Schema enforcement** catches bad extractions before they reach downstream systems |
| 3 | **Extraction quality scales** from regex to guided LLM to demonstrations |
| 4 | **Measure separately**: retrieval recall, field accuracy, schema compliance |
| 5 | **Production systems** use Pydantic models (LangChain) or component contracts (Haystack) |
| 6 | **Structured output** (JSON mode / function calling) > free-text + parsing |

### When to Use This Pattern

| Scenario | Recommendation |
|---|---|
| Converting incident reports to tickets | Retrieval-to-extraction |
| Extracting entities from contracts | Extraction with schema validation |
| Building structured knowledge graphs | Retrieve-extract-link |
| Log analysis and alerting | Retrieval + structured field extraction |

---
*Educational notebook -- NLP / Retrieval-to-Extraction Pipeline Portfolio.*